# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [3]:
import pandas as pd
import numpy as np


url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())


df = spotify_raw.drop_duplicates().copy()   # hay 2 filas idénticas repetidas

# Las métricas vienen como texto, quitar comas y pasar a número
metricas = ['Track Score', 'Spotify Streams', 'YouTube Views', 'YouTube Likes',
            'TikTok Views', 'Spotify Popularity', 'Shazam Counts']
for col in metricas:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

# Fecha año, mes, día
df['Release Date'] = pd.to_datetime(df['Release Date'], format='%m/%d/%Y')
df['Year']  = df['Release Date'].dt.year
df['Month'] = df['Release Date'].dt.month
df['Day']   = df['Release Date'].dt.day



Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [4]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

dim_tiempo = (df[['Release Date', 'Year', 'Month', 'Day']]
              .drop_duplicates()
              .sort_values('Release Date')
              .reset_index(drop=True))
dim_tiempo.insert(0, 'id_tiempo', range(1, len(dim_tiempo) + 1))

print("\ndim_tiempo:", dim_tiempo.shape)   # (1562, 5)
display(dim_tiempo.head())


dim_tiempo: (1562, 5)


,id_tiempo,Release Date,Year,Month,Day
0,1,1987-07-21,1987,7,21
1,2,1991-09-24,1991,9,24
2,3,1994-01-01,1994,1,1
3,4,1998-04-03,1998,4,3
4,5,1999-06-08,1999,6,8


In [5]:
# 2. Crear dim_pista
dim_pista = df[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista.insert(0, 'id_pista', range(1, len(dim_pista) + 1))
display(dim_pista.head())

,id_pista,Track,Artist,Explicit Track
0,1,MILLION DOLLAR BABY,Tommy Richman,0
1,2,Not Like Us,Kendrick Lamar,1
2,3,i like the way you kiss me,Artemas,0
3,4,Flowers,Miley Cyrus,0
4,5,Houdini,Eminem,1


In [6]:
# 3. Crear fact_streams
fact_streams = (df.merge(dim_tiempo[['id_tiempo', 'Release Date']], on='Release Date', how='left')
                  .merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left'))
fact_streams = fact_streams[['id_pista', 'id_tiempo'] + metricas]

display(fact_streams.head())
print(fact_streams.dtypes) 

,id_pista,id_tiempo,Track Score,Spotify Streams,YouTube Views,YouTube Likes,TikTok Views,Spotify Popularity,Shazam Counts
0,1,1519,725.4,3.904709e+08,8.427475e+07,1713126.0,5.332282e+09,92.0,2669262.0
1,2,1525,545.9,3.237039e+08,1.163470e+08,3486739.0,2.083390e+08,92.0,1118279.0
2,3,1487,538.4,6.013093e+08,1.225991e+08,2228730.0,3.369121e+09,92.0,5285340.0
3,4,1176,444.9,2.031281e+09,1.096101e+09,10629796.0,1.460373e+10,85.0,11822942.0
4,5,1550,423.3,1.070349e+08,7.737396e+07,3670188.0,NaN,88.0,457017.0


id_pista                int64
id_tiempo               int64
Track Score           float64
Spotify Streams       float64
YouTube Views         float64
YouTube Likes         float64
TikTok Views          float64
Spotify Popularity    float64
Shazam Counts         float64
dtype: object


In [7]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = (fact_streams.merge(dim_pista, on='id_pista', how='left')
                         .merge(dim_tiempo, on='id_tiempo', how='left'))
print(cubo_olap.shape) 

(4598, 16)


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

cuándo salió la canción (tiempo) y qué es (pista, artista y si es explícita). Todo lo demás son métricas.

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [8]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up (se agregan canciones al nivel de artista)
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum().sort_values(ascending=False)
display(respuesta_1.head(10))

# RETO 2: Todas las métricas de canciones lanzadas en 2024
# Operación OLAP: Slice (un solo valor en una dimensión: Year = 2024)
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
display(respuesta_2.head())

# RETO 3: Streams por Año y luego por Mes
# Operación OLAP: Drill-down (de Año a Año → Mes, más detalle)
respuesta_3 = cubo_olap.groupby(['Year', 'Month'])['Spotify Streams'].sum()
display(respuesta_3)

# RETO 4: Canciones explícitas Y lanzadas en 2024
# Operación OLAP: Dice (filtro en dos dimensiones: Explícita y Año)
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['Year'] == 2024)]
display(respuesta_4.head())

# RETO 5: Promedio de 'Track Score' por Año
# Operación OLAP: Roll-up (se sube de fecha completa a nivel Año)
respuesta_5 = cubo_olap.groupby('Year')['Track Score'].mean()
display(respuesta_5)

Artist
Bad Bunny         3.705483e+10
The Weeknd        3.694854e+10
Drake             3.496216e+10
Taylor Swift      3.447077e+10
Post Malone       2.613747e+10
Ed Sheeran        2.401490e+10
Ariana Grande     2.346499e+10
MUSIC LAB JPN     2.286669e+10
Olivia Rodrigo    1.972922e+10
Eminem            1.887888e+10
Name: Spotify Streams, dtype: float64

,id_pista,id_tiempo,Track Score,Spotify Streams,YouTube Views,YouTube Likes,TikTok Views,Spotify Popularity,Shazam Counts,Track,Artist,Explicit Track,Release Date,Year,Month,Day
0,1,1519,725.4,390470936.0,84274754.0,1713126.0,5.332282e+09,92.0,2669262.0,MILLION DOLLAR BABY,Tommy Richman,0,2024-04-26,2024,4,26
1,2,1525,545.9,323703884.0,116347040.0,3486739.0,2.083390e+08,92.0,1118279.0,Not Like Us,Kendrick Lamar,1,2024-05-04,2024,5,4
2,3,1487,538.4,601309283.0,122599116.0,2228730.0,3.369121e+09,92.0,5285340.0,i like the way you kiss me,Artemas,0,2024-03-19,2024,3,19
4,5,1550,423.3,107034922.0,77373957.0,3670188.0,NaN,88.0,457017.0,Houdini,Eminem,1,2024-05-31,2024,5,31
6,7,1445,407.2,900158751.0,308723145.0,4120760.0,5.349153e+08,86.0,9990302.0,Beautiful Things,Benson Boone,0,2024-01-18,2024,1,18


Year  Month
1987  7        1.879386e+09
1991  9        2.021910e+09
1994  1        1.810650e+09
1998  4        1.783399e+08
1999  6        1.405354e+09
                   ...     
2024  2        1.023077e+10
      3        1.263673e+10
      4        2.152035e+10
      5        1.005348e+10
      6        2.711085e+09
Name: Spotify Streams, Length: 191, dtype: float64

,id_pista,id_tiempo,Track Score,Spotify Streams,YouTube Views,YouTube Likes,TikTok Views,Spotify Popularity,Shazam Counts,Track,Artist,Explicit Track,Release Date,Year,Month,Day
1,2,1525,545.9,3.237039e+08,116347040.0,3486739.0,2.083390e+08,92.0,1118279.0,Not Like Us,Kendrick Lamar,1,2024-05-04,2024,5,4
4,5,1550,423.3,1.070349e+08,77373957.0,3670188.0,NaN,88.0,457017.0,Houdini,Eminem,1,2024-05-31,2024,5,31
7,8,1455,375.8,6.750792e+08,228382568.0,1439495.0,3.804584e+09,92.0,6063523.0,Gata Only,FloyyMenor,1,2024-02-02,2024,2,2
8,9,1558,355.7,1.653018e+09,NaN,NaN,NaN,NaN,NaN,Danza Kuduro - Cover,MUSIC LAB JPN,1,2024-06-09,2024,6,9
9,10,1543,330.6,9.067657e+07,32735244.0,988682.0,9.746562e+08,86.0,666302.0,BAND4BAND (feat. Lil Baby),Central Cee,1,2024-05-23,2024,5,23


Year
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    38.018868
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.711688
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [9]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: [Escribe aquí...]
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
display(respuesta_6.head())

# RETO 7: 'YouTube Views' por Artista y dentro de cada uno por Track
# Operación OLAP: Drill-down (de Artista a Artista → Canción)
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum()
display(respuesta_7.head(20))

# RETO 8: Sub-cubo de Diciembre 2023
# Operación OLAP: Dice (dos condiciones en la dimensión tiempo: Year y Month)
respuesta_8 = cubo_olap[(cubo_olap['Year'] == 2023) & (cubo_olap['Month'] == 12)]
display(respuesta_8.head())

# RETO 9: Total histórico de 'Spotify Streams' por Mes
# Operación OLAP: Roll-up (se colapsan años y días, solo queda el Mes)
respuesta_9 = cubo_olap.groupby('Month')['Spotify Streams'].sum().sort_values(ascending=False)
display(respuesta_9)

# RETO 10: Taylor Swift, NO explícitas, de 2024
# Operación OLAP: Dice (tres dimensiones: Artista, Explícita y Año)
respuesta_10 = cubo_olap[(cubo_olap['Artist'] == 'Taylor Swift') &
                         (cubo_olap['Explicit Track'] == 0) &
                         (cubo_olap['Year'] == 2024)]
display(respuesta_10)

,id_pista,id_tiempo,Track Score,Spotify Streams,YouTube Views,YouTube Likes,TikTok Views,Spotify Popularity,Shazam Counts,Track,Artist,Explicit Track,Release Date,Year,Month,Day
63,61,1021,184.8,1.811991e+09,1.516863e+09,8870768.0,6.470937e+09,79.0,3393920.0,Me Porto Bonito,Bad Bunny,1,2022-05-06,2022,5,6
88,86,1021,163.1,1.514240e+09,1.366181e+09,9358441.0,5.249864e+09,77.0,5133847.0,Titï¿½ï¿½ Me Pregu,Bad Bunny,1,2022-05-06,2022,5,6
102,100,1379,150.7,4.781509e+08,1.133797e+08,1312763.0,2.111289e+09,78.0,2255465.0,MONACO,Bad Bunny,1,2023-10-13,2023,10,13
174,171,1379,114.2,6.891815e+08,1.833815e+08,738123.0,4.874404e+08,83.0,930098.0,PERRO NEGRO,Bad Bunny,1,2023-10-13,2023,10,13
187,184,1021,110.7,1.421763e+09,9.458571e+08,5587691.0,2.251428e+08,78.0,3087489.0,Ojitos Lindos,Bad Bunny,0,2022-05-06,2022,5,6


Artist        Track                                                              
"XY"          Come and Say - Original Mix                                            415159148.0
$OHO BANI     ZEIT, DASS SICH WAS DREHT                                                6744201.0
$uicideboy$   ...And to Those I Love, Thanks for Sticking Around                     269460083.0
              1000 Blunts                                                             87680985.0
              Are You Going to See the Rose in the Vase, or the Dust on the Table      2640169.0
              Avalon                                                                  49001959.0
              For the Last Time                                                      149247747.0
              Matte Black                                                             82186979.0
              Not Even Ghosts Are This Empty                                          50747796.0
              The Thin Grey Line             

,id_pista,id_tiempo,Track Score,Spotify Streams,YouTube Views,YouTube Likes,TikTok Views,Spotify Popularity,Shazam Counts,Track,Artist,Explicit Track,Release Date,Year,Month,Day
61,59,1433,186.5,1384.0,222076.0,5430.0,8.811417e+09,NaN,NaN,Very Cute Melody by Marimba Tone (39813),mitsu sound,0,2023-12-31,2023,12,31
94,92,1413,155.9,649404552.0,433839212.0,2750086.0,1.060356e+07,83.0,1380069.0,La Diabla,Xavi,0,2023-12-02,2023,12,2
97,95,1415,153.1,359803580.0,369321334.0,1839225.0,8.740562e+08,86.0,1002835.0,BELLAKEO,Peso Pluma,1,2023-12-07,2023,12,7
131,129,1417,130.7,8327.0,NaN,NaN,7.331825e+07,NaN,522.0,Looking Like a Wow End of Year 2023,WasteLand,0,2023-12-10,2023,12,10
177,174,1412,112.6,606124553.0,166363445.0,1019744.0,2.960124e+07,86.0,1704574.0,LUNA,Feid,0,2023-12-01,2023,12,1


Month
1     3.652168e+11
5     1.779293e+11
11    1.734614e+11
10    1.672667e+11
3     1.638041e+11
7     1.503832e+11
6     1.484293e+11
4     1.462846e+11
8     1.426826e+11
9     1.361347e+11
2     1.333565e+11
12    1.016706e+11
Name: Spotify Streams, dtype: float64

,id_pista,id_tiempo,Track Score,Spotify Streams,YouTube Views,YouTube Likes,TikTok Views,Spotify Popularity,Shazam Counts,Track,Artist,Explicit Track,Release Date,Year,Month,Day
16,17,1512,297.6,395433400.0,104673836.0,2484083.0,207317000.0,89.0,1210029.0,Fortnight (feat. Post Malone),Taylor Swift,0,2024-04-18,2024,4,18
866,861,1512,52.3,166168092.0,16423009.0,244859.0,7202700.0,78.0,116673.0,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0,2024-04-18,2024,4,18
1013,1007,1512,47.8,175613480.0,15237337.0,222750.0,1510800.0,84.0,67886.0,"So Long, London",Taylor Swift,0,2024-04-18,2024,4,18
1167,1159,1512,44.0,161916445.0,13509432.0,180327.0,NaN,83.0,47529.0,My Boy Only Breaks His Favorite Toys,Taylor Swift,0,2024-04-18,2024,4,18
1270,1260,1512,42.1,165611453.0,9873283.0,96083.0,6383500.0,78.0,79964.0,Guilty as Sin?,Taylor Swift,0,2024-04-18,2024,4,18
2483,2454,1513,28.6,97601615.0,5528058.0,78517.0,17893741.0,87.0,NaN,So High School,Taylor Swift,0,2024-04-19,2024,4,19
2516,2486,1512,28.3,106201248.0,5521347.0,70034.0,NaN,NaN,24502.0,The Alchemy,Taylor Swift,0,2024-04-18,2024,4,18
2527,2497,1512,28.2,250430303.0,NaN,NaN,NaN,NaN,NaN,I Can Do It With a Broken Heart,Taylor Swift,0,2024-04-18,2024,4,18
2775,2744,1513,26.7,88244010.0,7381186.0,96742.0,14644000.0,NaN,32559.0,imgonnagetyouback,Taylor Swift,0,2024-04-19,2024,4,19
2815,2784,1512,26.5,105424914.0,7227379.0,113318.0,2500000.0,NaN,21372.0,Fresh Out The Slammer,Taylor Swift,0,2024-04-18,2024,4,18
